# 03 · Join Sofascore + Capology — Italy Serie A 25/26 (snapshot 20260428)

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2025/26 de la Serie A italiana**.

⚠️ **Nota sobre el snapshot:** la temporada 25/26 está aún en curso. Se trabaja con
una foto fija de Sofascore (`df_italy_2526_snapshot_20260428.csv`). Este notebook
deberá reejecutarse con los datos definitivos cuando finalice la liga, generando
entonces el master sin sufijo de fecha (`master_italy_2526.csv`).

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_italy_2526_snapshot_20260428.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_italy_2526.csv').copy()

print(f'Sofascore (snapshot):  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:              {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore (snapshot):  561 jugadores | 117 columnas
Capology:              607 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   inter
   milan

En Capology pero no en Sofascore:
   ac milan
   inter milan


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [6]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'ac milan':'milan',
            'inter milan':'inter'

}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')

✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [7]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 487/561 (86.8%)
Sin emparejar: 74


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [8]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          12
Revisión media    (0.75 ≤ score < 0.90):   9
Revisión estricta (0.50 ≤ score < 0.75):   35
Revisión muy est. (score < 0.50):           18


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [9]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
0,Benjamin Pavard,Inter,benjamin pavard,1.000
1,Samuel Chukwueze,Milan,samuel chukwueze,1.000
41,Álex Jiménez,Milan,alex jimenez,1.000
20,Albert Guðmundsson,Fiorentina,albert gudmundsson,0.971
14,Łukasz Skorupski,Bologna,lukasz skorupski,0.968
21,Enrico Delprato,Parma,enrico del prato,0.968
12,Jan Ziółkowski,Roma,jan ziolkowski,0.963
9,Evan Ndicka,Roma,evan n dicka,0.957
50,Þórir Jóhann Helgason,Lecce,thorir johann helgason,0.952
47,Rafael Obrador,Torino,rafa obrador,0.923


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [10]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
32,Mikael Ellertsson,Genoa,mikael egill ellertsson,0.850
40,Kostas Tsimikas,Roma,konstantinos tsimikas,0.833
63,Henrik Wendel Meister,Pisa,henrik meister,0.800
16,Ange Bonny,Inter,ange yoan bonny,0.800
2,Yann Bisseck,Inter,yann aurel bisseck,0.800
29,Michel Adopo,Cagliari,michel ndary adopo,0.800
68,Alessandro Di Pardo,Cagliari,alessandro deiola,0.778
37,Nicolas-Gerrit Kühn,Como,nicolas kuhn,0.774
54,Marcus Pedersen,Torino,marcus holmgren pedersen,0.769


In [11]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['alessandro di pardo'

]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')

Aceptados: 8 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [12]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
7,Marc Kempf,Como,marc oliver kempf,0.741
22,Andrea Ghion,Sassuolo,andrea pinamonti,0.714
30,Adrian Benedyczak,Parma,adrian bernabe,0.710
34,Moatasem Al-Musrati,Hellas Verona,al musrati,0.690
53,Gift Orban,Hellas Verona,gift emmanuel orban,0.690
38,Sulemana,Cagliari,ibrahim sulemana,0.667
46,Nicolae Stanciu,Genoa,nicola leali,0.667
15,Alexsandro Amorim,Genoa,alessandro marcandalli,0.667
8,Frank Anguissa,Napoli,andre zambo anguissa,0.647
64,Matteo Lavelli,Inter,matteo darmian,0.643


In [13]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['marc kempf',
                    'moatasem al musrati',
                    'gift orban',
                    'sulemana',
                    'frank anguissa',
                    'manu kone',
                    'suzuki zion'

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')

Aceptados del nivel bajo: 7


### 7.4 Revisión muy estricta (score < 0.50)

Candidatos con muy baja similitud. Por defecto ninguno se acepta.
Añadir a `ACCEPT_VERY_LOW_FUZZY` los que se confirmen manualmente.

In [14]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
18,Ademola Lookman,Atalanta,kamaldeen sulemana,0.485
49,Ioan Vermesan,Hellas Verona,martin frese,0.480
28,Hernani,Parma,franco carboni,0.476
43,Olaf Gorter,Lecce,oumar ngom,0.476
10,Valentín Castellanos,Lazio,alessio furlanetto,0.474
56,Alessandro Romano,Roma,lorenzo venturino,0.471
65,Louis Buffon,Pisa,rosen bozhinov,0.462
66,Tomás Esteves,Pisa,calvin stengs,0.462
70,Tjaš Begić,Parma,sascha britschgi,0.462
60,Giovanni Bonfanti,Pisa,simone canestrelli,0.457


In [15]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')

Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [16]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 511/561 (91.1%)
Sin salario:     50


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [17]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 50


,player,team,minutesPlayed,appearances,goals,assists
0,Ademola Lookman,Atalanta,797,12,2,0
1,Ciro Immobile,Bologna,191,6,0,0
2,Paul Mendy,Cagliari,133,5,2,0
3,Alessandro Di Pardo,Cagliari,60,3,0,0
4,Nicolò Cavuoti,Cagliari,52,4,0,0
5,Andrea Le Borgne,Como,1,1,0,0
6,Franco Vazquez,Cremonese,529,15,1,2
7,Dennis Johnsen,Cremonese,390,11,1,1
8,Dachi Lordkipanidze,Cremonese,28,1,0,0
9,Pablo Mari,Fiorentina,809,11,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [18]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Atalanta  —  SF sin salario:


,player,minutesPlayed
0,Ademola Lookman,797


  CG plantilla completa:


,player,player_norm
0,Ben Godfrey,ben godfrey
1,Berat Djimsiti,berat djimsiti
2,Charles De Ketelaere,charles de ketelaere
3,Daniel Maldini,daniel maldini
4,Davide Zappacosta,davide zappacosta
5,Éderson,ederson
6,El Bilal Touré,el bilal toure
7,Francesco Rossi,francesco rossi
8,Giacomo Raspadori,giacomo raspadori
9,Gianluca Scamacca,gianluca scamacca



  Bologna  —  SF sin salario:


...

,player,player_norm
0,Adrien Tamèze,adrien tameze
1,Alberto Paleari,alberto paleari
2,Ali Dembélé,ali dembele
3,Alieu Njie,alieu njie
4,Ardian Ismajli,ardian ismajli
5,Cesare Casadei,cesare casadei
6,Ché Adams,che adams
7,Cristiano Biraghi,cristiano biraghi
8,Cyril Ngonge,cyril ngonge
9,Duván Zapata,duvan zapata



  Udinese  —  SF sin salario:


,player,minutesPlayed
0,Saba Goglichidze,422


  CG plantilla completa:


,player,player_norm
0,Abdoulaye Camara,abdoulaye camara
1,Adam Buksa,adam buksa
2,Alessandro Nunziante,alessandro nunziante
3,Alessandro Zanoli,alessandro zanoli
4,Arthur Atta,arthur atta
5,Branimir Mlacic,branimir mlacic
6,Brenner,brenner
7,Christian Kabasele,christian kabasele
8,Daniele Padelli,daniele padelli
9,Hassane Kamara,hassane kamara


In [19]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {

}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')

Matches manuales definidos: 0


In [20]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')


Tras matches manuales: 511/561 (91.1%)


## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

⚠️ Nombre con sufijo `_snapshot_20260428` para diferenciar del master definitivo
que se generará al cierre de la temporada (`master_italy_2526.csv`).

In [21]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_italy_2526_snapshot_20260428.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_italy_2526_snapshot_20260428.csv
   Jugadores totales:  561
   Con salario:        511
   Sin salario (NaN):  50
   Columnas:           122
